# 2026 Spring "EE50045, QU50001: Introductory Quantum Mechanics"

###### Update history
###### 2021. 01. 01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code.
###### 2024. 08. 07 : Minsu Jeong,  KAIST Electrical Engineering, Write Kronig-Penny code based on TISE. 
###### 2025. 03. 27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization (plotly) and the description. 
###### 2025. 09. 30 : Minsu Jeong,  KAIST Electrical Engineering, added effective mass

###### ref
###### - Kittel, Introduction to Solid State Physics, 8ed (John Wiley & Sons, Inc) p169
###### - Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012), p63
###### - John R. Hiller, Quantum Mechanics Simulations The Consortium for upper-Level Physics Software, (John Wiley & Sons)


# Kronig-Penny model
---
It is a simplified but powerful approach to understanding electron behavior in periodic potentials, which are crucial for explaining the electronic structure of crystalline solids.

In solids, electrons experience a periodic potential created by the regularly arranged atoms in a crystal lattice.
The Kronig-Penny model approximates this potential as a one-dimensional series of alternating potential wells and barriers.
Despite its simplicity, it successfully captures essential features like allowed and forbidden energy bands.

The potential assumed in the Kronig-Penny model can be visualized as:

```bash
Potential V(x)

   V0 ┌───┐     ┌───┐     ┌───┐
      │   │     │   │     │   │
0─────┘   └─────┘   └─────┘   └───────► x
  ... | b |  a  | b |  a  | ...
          <─────────>
             period
```
- **a:** Well width, region of zero potential
- **b:** Barrier width, region of potential $ V_0 $


The Kronig-Penny model is analytically described by solving the Schrödinger equation for electrons subjected to a periodic square potential:


---
## Derivation Using Bloch Waves
The Schrödinger equation for an electron in a periodic potential $ V(x) $ is given by:

$
-\frac{\hbar^2}{2m}\frac{d^2 \psi(x)}{dx^2} + V(x)\psi(x) = E\psi(x)
$

For periodic potentials, Bloch's theorem states that the wavefunction can be written in the form:

$
\psi_k(x) = e^{ikx}u_k(x)
$

where $ u_k(x) $ is a periodic function with the same periodicity as the potential, i.e., $ u_k(x+a+b)=u_k(x) $.

Considering a piecewise solution within each region (well and barrier), we apply continuity conditions for both the wavefunction and its derivative at the interfaces. Solving these boundary conditions and applying Bloch's theorem yields the transcendental Kronig-Penny equation.


The final transcendental equation, known as the Kronig-Penny equation, can be expressed as:

$
\cos(k(a + b)) = \cos(k a) \cosh(q b) + \frac{q^2 - k^2}{2 k q} \sin(k a) \sinh(q b)
$

where:
- $ k $ is the electron wavevector inside the wells, given by $ k = \sqrt{\frac{2mE}{\hbar^2}} $.
- $ q $ is the electron wavevector inside the barriers, given by $ q = \sqrt{\frac{2m(V_0 - E)}{\hbar^2}} $, with $ V_0 $ as the barrier height.
- $E$ is the electron energy.


```python
※The dimension of Hamiltonian will be [ngridx, ngridx], where ngridx denotes the number of points in real space.
```

In [ ]:
'''
Update history
2021. 01. 01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code.
2024. 08. 07 : Minsu Jeong,  KAIST Electrical Engineering, Write Kronig-Penny code based on TISE. 
2025. 03. 27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization (plotly) and the description. 
2025. 09. 30 : Minsu Jeong,  KAIST Electrical Engineering, added effective mass

ref
- Kittel, Introduction to Solid State Physics, 8ed (John Wiley & Sons, Inc) p169
- Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012), p63
- John R. Hiller, Quantum Mechanics Simulations The Consortium for upper-Level Physics Software, (John Wiley & Sons)

'''

###############################################################################
# Imports
###############################################################################
import numpy as np
import numpy.linalg as lin
import math
import ipywidgets as widgets
from ipywidgets import interact_manual, IntSlider, FloatSlider, Dropdown, Layout
from scipy.signal import argrelextrema
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Import physical constants from scipy
from scipy.constants import physical_constants

# Set numpy print options for debugging (optional)
np.set_printoptions(threshold=784, linewidth=np.inf)

###############################################################################
# Global Constants and Unit Conversions
###############################################################################
# Constant and unit conversion
# 1 Bohr = 0.52917721067 Angstrom, so 1 Angstrom = 1/0.52917721067 Bohr ~ 1.88973
hbar = physical_constants["reduced Planck constant"][0]

bohr_radius     = physical_constants['Bohr radius'][0]  # in miter
angstrom_radius = 1e-10                                 # in miter
eff_ang2bohr        = 1e-10 / bohr_radius  # ~1.88973

hartree_energy  = physical_constants['Hartree energy'][0]   # in Jule
ev_energy       = physical_constants['electron volt'][0]    # in Jule
eff_har2ev          = hartree_energy / ev_energy  # ~27.21140795

# Global variable for Laplacian finite difference accuracy
laplacianDim = 3

###############################################################################
# Laplacian Functions (Finite Difference Method)
###############################################################################
def laplacianCoeff(laplacianDim):
    """
    Compute the coefficients for the Laplacian operator using the Finite Difference Method (FDM).

    This function constructs the finite difference coefficients for the second derivative 
    using central difference formulas. The method is based on solving a linear system 
    that relates the coefficients to the derivatives of a function at grid points.
    """
    size = 2 * laplacianDim + 1
    a = np.zeros((size, size))
    c = np.zeros(size)
    c[2] = math.factorial(2)  # For second derivative, 2! = 2
    for i in range(size):
        for j in range(size):
            a[i, j] = (j - laplacianDim) ** i
    inva = lin.inv(a)
    b = inva @ c  # b = a^-1 * c
    lcoeff = b[laplacianDim:size]
    return lcoeff

def laplacian(ngridx, laplacianDim, dx, num_well, is_kroing):
    """
    Construct the Laplacian operator matrix for a 1D grid using the Finite Difference Method.
    Uses periodic boundary conditions.
    """

    ##

    lcoeff = laplacianCoeff(laplacianDim)
    loprt = np.zeros((ngridx, ngridx))

    for i in range(ngridx):
        for j in range(-laplacianDim, laplacianDim + 1):
            k = i + j

            # Dirichlet (Closed) Boundary Condition
            if not is_kroing: 
                if  k < 0 or k >= ngridx: continue  

            # Periodic Boundary condition
            else:
                if  k >= ngridx : k -= ngridx
                elif k < 0      : k += ngridx

            loprt[i, k] = lcoeff[abs(j)] / (dx ** 2)
            # to get full band structrue for all k space,
            # multiply phase term ( exp(ik) or exp(-ik))

    return loprt

###############################################################################
# Potential and Hamiltonian Functions
###############################################################################
def potential(ngridx, num_well, pot_shape, pot_height_har=25, width_barrier_bohr=2, width_well_bohr=2, is_kroing=False):
    """
    Set up the potential for the Kronig-Penny model.
    
    In each cell (period), the barrier region has width 'width_barrier' and 
    the well region has width 'width_well'. The potential barrier height is pot_height_har, 
    and the well region is 0.
    
    Parameters:
    -----------
    ngridx      : int
        Number of grid points.
    num_well    : int
        Number of cells (periods).
    pot_height_har      : float
        Barrier height in Hartree.
    width_barrier_bohr  : float
        Barrier width.
    width_well_bohr     : float
        Well width.
    
    Returns:
    --------
    pot_grid : ndarray
        The potential at each grid point (in Hartree).
    """

    ngridx_per_cell = ngridx // num_well
    pot_grid = np.zeros(ngridx)

    frac_well       = width_well_bohr / (width_barrier_bohr + width_well_bohr)
    frac_barrier    = 1 - frac_well
    
    i_elec_lr    = num_well_lr if not is_kroing else 0

    

    ### Set up offset (for center aligned potential) & length of well
    '''
            barrier  well
           <-------><---->
           ┌-------┐     ┌----- height
           │       │     │
           │       │     │
        ---┘       └-----┘      0
        -------┼---------------->
             center
            <-->
           offset
               
    '''
    # Square well & Triangualr well
    if pot_shape == 0:
        offset      = 0.5 * frac_barrier * ngridx_per_cell if not is_kroing else 0
        len_well    = frac_well * ngridx_per_cell

    # Parabolic well (Y-shaped (convex) & U-shaped (concave))
    elif pot_shape == 1 or pot_shape == 2:
        offset      = 0
        len_well    = ngridx_per_cell 

    # Coulombic
    elif pot_shape == 3:
        if not is_kroing:
            offset      = - (num_well_lr - 0.5) * ngridx_per_cell
            len_well    = 2* num_well_lr * ngridx_per_cell 
        else:
            offset      = 0
            len_well    = ngridx_per_cell 



    ### Set up potential
    for i in range( i_elec_lr, num_well - i_elec_lr):
        start   = int(i * ngridx_per_cell + offset)
        end     = int(i * ngridx_per_cell + offset + len_well)

        # Square well
        if pot_shape == 0: 
            pot_grid[start:end] -= pot_height_har

        # Triangualr well
        elif pot_shape == 1:
            for ii in range (start, end - 1):
                pot_grid[ii] -=  ((ii - start) / (end + 1 - start)) * pot_height_har

        # Parabolic (concave) (U-shaped well)
        elif pot_shape == 2:
            for ii in range (start, end):
                pot_grid[ii] -= (1 - ( ( ii - (start + end)/2 ) / ((end - start)/2 ) )**2 ) * pot_height_har

        # Coulomb (legacy)
        # # Parabolic (convex) (Approximation of Coulomic well, Y-shaped well, 1-r**2)
        # elif pot_shape == 3:
        #     for ii in range (start, (end + start)//2):
        #         pot_grid[ii] = ( 1 - ((ii - start) / ((end - start)/2 ))**2 ) * pot_height_har
        #     for ii in range ((end + start)//2, end):
        #         pot_grid[ii] = ( 1 - ((ii - end) / ((end - start)/2 ))**2 ) * pot_height_har

        # Coulomb (soft-core)
            '''
            Used soft core to avoid -inf.

            V(r) = - Z / (r^a + k^a)^(1/a)

            r : radial position
            k : cutoff parameter
            a : order  parameter

            ref - Phys. Rev. A, 80, 032507 (2009)
            '''
        elif pot_shape == 3:
            alpha = 8       # order of Coulomb potential
            soft_core = 1   # level of soft core (cutoff) of Coulomb potential

            for ii in range (start, end ):
                coulombic_dacay = 1 / np.power( np.abs(soft_core**alpha + (ii - (end + start)//2)**alpha) , 1/alpha )
                pot_coulombic_decay = coulombic_dacay * pot_height_har

                pot_grid[ii] -= pot_coulombic_decay


        # Custom potential   Design your potential :)
        elif pot_shape == 4:
            for ii in range (start, end ):
                pot_grid[ii] -= 0


    return pot_grid

def hamiltonian(ngridx, laplacianDim, dx, num_well, pot_shape, pot_height_har=25, 
                width_barrier_bohr=2, width_well_bohr=2, is_kroing=False):
    """
    Construct the Hamiltonian matrix H = -1/2 * Laplacian + Potential.
    """
    pot_op  = np.zeros((ngridx, ngridx))
    pot     = potential(ngridx, num_well, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, is_kroing)
    laplacian_op = laplacian(ngridx, laplacianDim, dx, num_well, is_kroing)
    np.fill_diagonal(pot_op, pot)

    ### Effective mass = 1, hbar = 1 (atomic unit)
    ham_op = -laplacian_op / 2. + pot_op
    return ham_op

def solve_tise(ngridx, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, dx, num_well, is_kroing):
    """
    Solve the time-independent Schrödinger equation and return eigenvalues and eigenvectors.
    """
    ham_op = hamiltonian(ngridx, laplacianDim, dx, num_well, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, is_kroing)
    eigval, eigvec = np.linalg.eigh(ham_op)
    return eigval, eigvec

###############################################################################
# Plot Functions Using Plotly
###############################################################################
def plot_supercell(eigval, eigvec, ngridx, box_bohr,   
                                pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, num_bands, num_well, is_kroing):
    """
    Plot eigenstates and potential using Plotly.
    """
    # x coordinate in Angstrom
    x = np.linspace(-box_bohr/2, box_bohr/2, ngridx) / eff_ang2bohr  # Convert from Bohr to Angstrom

    # Determine indices for eigenstates (min and max for each i_band)
    i_eig = []
    list_color = ['red', 'green', 'blue', 'magenta', 'yellow', 'cyan']
    list_line = ['solid', 'dash']

    pot = potential(ngridx, num_well, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, is_kroing)

    num_well -= 2 * num_well_lr if not is_kroing else 0


    # Create subplots
    fig = make_subplots(rows=1, cols=2, column_widths=[0.75, 0.25],
                        subplot_titles=("Eigenstates & Potential", ""))

    ### Left subplot: Eigenstates and potential
    for i in range(num_bands):
        # Each band has eigenvalues
        #       i'th band        index of a wavefunction
        iemin = i * num_well + np.argmin(eigval[i * num_well : (i + 1) * num_well])
        iemax = i * num_well + np.argmax(eigval[i * num_well : (i + 1) * num_well])

        if not is_kroing :
            if (0 < eigval[iemin]) or (0 < eigval[iemax]) :
                continue
            
        # i_eig.append(iemin)
        # i_eig.append(iemax)

        ieall = np.arange(i * num_well, (i+1) * num_well)
        i_eig.extend(ieall)


    for i in i_eig:
        eigstate = (eigvec[:, i] - eigvec[0, i]) * 10 + np.ones(ngridx) * eigval[i] * eff_har2ev    # normalize
        fig.add_trace(go.Scatter(x=x, y=eigstate,
                                 mode='lines',
                                 line=dict(color=list_color[(i // num_well) % len(list_color)],
                                        #    dash=list_line[((i + 1) % num_well - 1)%2]
                                           ),
                                 name=f"Eig {i}"),
                      row=1, col=1)
        fig.add_trace(go.Scatter(x=[x[0], x[-1]],
                                 y=[eigval[i] * eff_har2ev, eigval[i] * eff_har2ev],
                                 mode='lines',
                                 line=dict(color='gray', dash='dot'),
                                 showlegend=False),
                      row=1, col=1)

    # Plot potential
    
    fig.add_trace(go.Scatter(x=x, y=pot * eff_har2ev,
                             mode='lines',
                             line=dict(color='black'),
                             name="Potential"),
                  row=1, col=1)

    fig.update_xaxes(title_text="Position [Angstrom]", row=1, col=1)
    y_max_plot = 0.1 * pot_height_har * eff_har2ev if not is_kroing else eigval[i_eig[-1]] * 1.1 * eff_har2ev
    y_min_plot = - 1.1 * pot_height_har * eff_har2ev
    fig.update_yaxes(title_text="Energy [eV]", range=[y_min_plot, y_max_plot], row=1, col=1)

    ### Right subplot
    if is_kroing:
        for i in range(len(i_eig) // 2):
            y_min = eigval[i * num_well] * eff_har2ev
            y_max = eigval[(i + 1) * num_well - 1] * eff_har2ev
            fig.add_shape(type="rect",
                        xref="x2 domain", yref="y2",
                        x0=0, x1=1, y0=y_min, y1=y_max,
                        fillcolor=list_color[i % len(list_color)],
                        opacity=0.5, line_width=0,
                        row=1, col=2)
            fig.add_annotation(x=0.4, y=(y_min + y_max) / 2,
                            xref="x2", yref="y2",
                            text=f"Min: {y_min:.2f} eV<br>Max: {y_max:.2f} eV",
                            showarrow=False,
                            font=dict(size=10, color="black"),
                            xanchor="left",
                            row=1, col=2)
    else:
        for i_band in range(num_bands):
            idx = np.arange(i_band * num_well, (i_band + 1) * num_well)

            if eigval[idx[-1]] < 0:
                fig.add_trace(go.Scatter(
                    x=idx,
                    y=(eigval[idx] * eff_har2ev),
                    mode='markers',
                    marker=dict(size=6, color=list_color[i_band % len(list_color)]),
                    name=f'Band {i_band+1}'
                ), row=1, col=2)
    
    fig.update_xaxes(title_text="Occurance", row=1, col=2)
    fig.update_yaxes(title_text="Energy [eV]", range=[y_min_plot, y_max_plot], row=1, col=2)

    fig.update_layout(title_text="Eigenstates, Potential", width=1000, height=600)
    fig.show()

def solve_and_plot_analytic(eigval, pot_height_har, width_barrier_bohr, width_well_bohr, num_bands, box_bohr, num_well):
    """
    Solve the analytic equation of the Kronig-Penny model and plot the resulting E–k diagram using Plotly.
    Here, we set a = well width and b = barrier width (converted to Bohr).
    """
    list_color = ['red', 'green', 'blue', 'magenta', 'yellow', 'cyan']


    ######################
    ### Calculation ######
    ######################

    energy = np.linspace(1e-3, eigval[(num_bands+4)*num_well - 1], ngirde, endpoint=True)

    ### Effecitve mass = 1 , hbar = 1 (atomic unit)
    k_val = np.sqrt(2 * energy + 0j)        
    q_val = np.sqrt(2 * (pot_height_har - energy) + 0j)
    
    # In analytic solution, let a = well width and b = barrier width (convert Angstrom -> Bohr)
    a_val = width_well_bohr  # well width in Bohr
    b_val = width_barrier_bohr     # barrier width in Bohr
    
    functional = ((q_val**2 - k_val**2)/(2*q_val*k_val)) * np.sinh(q_val*b_val) * np.sin(k_val*a_val) \
                 + np.cosh(q_val*b_val) * np.cos(k_val*a_val)
    
    # Find min, max eigenvalues of each band
    e_low, e_high = [], []
    for i in range(len(functional) - 1):
        if (functional[i] < 1 and functional[i+1] > 1) or (functional[i] > 1 and functional[i+1] < 1):
            e_low.append(np.real(k_val[i]**2 / 2) - pot_height_har)
        if (functional[i] < -1 and functional[i+1] > -1) or (functional[i] > -1 and functional[i+1] < -1):
            e_high.append(np.real(k_val[i]**2 / 2) - pot_height_har)
    e_low.sort(); e_high.sort()


    ######################
    ### Visualization ####
    ######################

    fig = make_subplots(rows=1, cols=2, subplot_titles=("Analytic Functional", "E–k Diagram"))
    
    fig.add_trace(go.Scatter(
        x=np.real(k_val * a_val / np.pi),
        y=np.real(functional),
        mode='lines',
        name="Functional",
        line=dict(color='black')
    ), row=1, col=1)

    # possible solution (btw -1 ~ 1)
    fig.add_hline(y=1, line=dict(color='gray', dash='dot'), row=1, col=1)
    fig.add_hline(y=-1, line=dict(color='gray', dash='dot'), row=1, col=1)
    fig.add_shape(type="rect", xref="x1", yref="y1",
                  x0=np.min(np.real(k_val * a_val / np.pi)),
                  x1=np.max(np.real(k_val * a_val / np.pi)),
                  y0=1, y1=2, fillcolor="gray", opacity=0.2, layer="below",
                  row=1, col=1)
    fig.add_shape(type="rect", xref="x1", yref="y1",
                  x0=np.min(np.real(k_val * a_val / np.pi)),
                  x1=np.max(np.real(k_val * a_val / np.pi)),
                  y0=-2, y1=-1, fillcolor="gray", opacity=0.2, layer="below",
                  row=1, col=1)
    
    fig.update_xaxes(title_text="Ka/π", row=1, col=1)
    fig.update_yaxes(title_text="Functional", range=[-2, 2], row=1, col=1)
    
    kl = np.linspace(0, np.pi, 100)
    for i in range(num_bands):
        if i < len(e_high) and i < len(e_low):
            weight = np.real((e_high[i] - e_low[i]) / 2)
            shift = np.real((e_high[i] + e_low[i]) / 2)
            cos_func = (weight * (-np.cos(kl)) + shift) * eff_har2ev
            fig.add_trace(go.Scatter(
                x=kl,
                y=np.real(cos_func),
                mode='lines+markers',
                name=f'Band {i+1}',
                marker=dict(color=list_color[i % len(list_color)])
            ), row=1, col=2)
            center_y = (np.real(e_low[i]) + np.real(e_high[i])) / 2 * eff_har2ev
            fig.add_annotation(
                x=0.8 * np.pi, y=center_y,
                xref="x2", yref="y2",
                text=f"{np.real(e_low[i]*eff_har2ev):.2f} eV<br> {np.real(e_high[i]*eff_har2ev):.2f} eV",
                showarrow=False,
                font=dict(size=10, ),#color=list_color[i % len(list_color)]),
                xanchor="left"
            )
    fig.update_xaxes(title_text="k(a+b)", row=1, col=2)
    fig.update_yaxes(title_text="Energy [eV]", row=1, col=2)
    
    fig.update_layout(title_text="Analytic Solutions (Bloch wave)", width=1000, height=600, showlegend=True)
    fig.show()



###############################################################################
# Simulation option
###############################################################################


num_well_lr = 4 # left and right region to avoid the effect of the box 
num_well_kriong = 15    
'''
if num_well >= num_well__kroing, 
the calculation will be for the Kronig-Penny model 
with periodic boundary conditions, 
otherwise, the calculation will be for a model 
with a finite number of potential wells inside an infinite potential box.

'''

ngirde = 2500   # resolution of energy for analytic solution
ngridx_per_cell = 251   # number of grid points per cell




###############################################################################
# Interactive Widget
###############################################################################
slider_layout = Layout(width='400px')
label_style = {'description_width': '150px'}

@interact_manual(
    num_bands=IntSlider(min=1, max=8, step=1, value=2, 
                        description='# of Bands:', style=label_style, layout=slider_layout),
    
    pot_shape = widgets.Dropdown(
                        options=[
                            ('Square Well', 0),
                            ('Triangular Well', 1),
                            ('Parabolic Well', 2),
                            ('Coulombic well', 3),
                            ('Custom potential', 4),
                        ],
                        value=0,description='Potential Shape: ', style=label_style, layout=slider_layout
                    ),

    pot_height_eV=FloatSlider(min=0.0, max=100, step=0.1, value=3, 
                        description='Barrier Height [eV]:', style=label_style, layout=slider_layout
                    ),

    width_barrier_ang=FloatSlider(min=0.0, max=1000, step=0.1, value=3, 
                        description='Barrier Width [Ang]:', style=label_style, layout=slider_layout
                    ),
    
    width_well_ang=FloatSlider(min=0.0, max=1000, step=0.5, value=3, 
                        description='Well Width [Ang]:', style=label_style, layout=slider_layout
                    ),

    num_well = IntSlider(min=1, max=num_well_kriong, step=1, value=3, 
                        description='# of potential well:', style=label_style, layout=slider_layout
                    ),
    
    m_eff    = widgets.Dropdown(options=[
                                ('GaAs : 0.07',  0.07),
                                ('Silicon : 1.08', 1.08),
                                ('Germanium : 0.55', 0.55),
                                ('Bare mass : 1.00',  1.00),],
                        value=1.00, description='Effective mass', 
                        readout=True, style=label_style, layout=slider_layout
                    ),
    )

def main(pot_shape, m_eff, pot_height_eV, width_barrier_ang, width_well_ang, num_well, num_bands):
    
    is_kroing       = True  if num_well >= num_well_kriong else False
    # Determine whether to use 
    # (1) periodic boundary condition (PBC), which means we simulate Kronig-Penny model,
    # or (2) Dirichlet boundary condition (closed box), which menas we simulate in the closed box.

    num_well_total  = 6     if is_kroing else num_well + 2 * num_well_lr # Number of cells (wells)
    # Add left and right region to avoid the effect of the box. 
    # No addition at periodic boundary condition (Kronig-Penny model)

    ngridx  = ngridx_per_cell * num_well_total
    if pot_shape == 0:
        box_ang = num_well_total * (width_barrier_ang + width_well_ang)
    else:
        box_ang = num_well_total * (width_well_ang)
    

    ### Effective mass atomic unit conversion
    global eff_bohr
    global eff_ang2bohr
    global eff_har
    global eff_har2ev
    
    eff_bohr = bohr_radius / m_eff
    eff_ang2bohr = 1e-10 / eff_bohr
    eff_har = hartree_energy * m_eff
    eff_har2ev = eff_har / ev_energy
    
    box_bohr            = box_ang           * eff_ang2bohr
    pot_height_har      = pot_height_eV     / eff_har2ev
    width_well_bohr     = width_well_ang    * eff_ang2bohr 
    width_barrier_bohr  = width_barrier_ang * eff_ang2bohr
    
    dx = box_bohr / ngridx

    # print("-"*50,readme,"\n","-"*50)

    # Solve the time-independent Schrödinger equation
    print('Solving Schrödinger equation')
    eigval, eigvec = solve_tise(ngridx, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, dx, num_well_total, is_kroing)
    
    # Plot eigenstates & potential
    print('Visualizing')
    plot_supercell(eigval, eigvec, ngridx, box_bohr, pot_shape, pot_height_har, width_barrier_bohr, width_well_bohr, num_bands, num_well_total, is_kroing)
    print(eigval*eff_har2ev)

    if (is_kroing) and (pot_shape == 0) :
        print('Solving analytic equation of Kronig-Penny model and Visualizing the solution')
        solve_and_plot_analytic(eigval, pot_height_har, width_barrier_bohr, width_well_bohr, num_bands, box_bohr, num_well_total)

    print("Done!")


if __name__ == "__main__":
    readme = r"""
Potential V(x)

   V0 ┌───┐     ┌───┐     ┌───┐
      │   │     │   │     │   │
0─────┘   └─────┘   └─────┘   └───────► x
  ... | b |  a  | b |  a  | ...
          <─────────>
             period
    
    Barrier Height  = V₀ 
    Barrier Width   = b
    Well Width      = a
    Period of cell  = a + b 

※ If you set up [# of potential well] as maximum (=num_well_kriong),
    Simulator will use periodic boundary condition (PBC), which means it simulate Kronig-Penny model.
    """
    print("-"*50,readme,"\n","-"*50)

    pass
